<a href="https://colab.research.google.com/github/Gayathri-rfr/RepoWalker_VectorRAG/blob/main/GithubvectorlessRepoWalker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implementing vectorless rag


> Convert the github repo code/content to json/graph inorder to find semantic patterns instead of embeddings(vector RAG)



In [ ]:
!pip install --quiet langchain langchain-openai langgraph pydantic

In [ ]:
import os
import shutil
from google.colab import userdata
from langchain_openai import ChatOpenAI

# 1. Secure Token Retrieval
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
    print("GITHUB_TOKEN loaded.")
except userdata.SecretNotFoundError:
    raise ValueError("GITHUB_TOKEN not found in Colab Secrets sidebar.")

# 2. Configuration
REPO_OWNER = "tiangolo"
REPO_NAME = "typer"
LOCAL_DIR = f"./{REPO_NAME}"
GITHUB_MODELS_BASE = "https://models.inference.ai.azure.com"

# 3. Initialize LLM (Using GPT-4o-mini via GitHub Models)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key=GITHUB_TOKEN,
    openai_api_base=GITHUB_MODELS_BASE
)
print("LLM engine ready.")

GITHUB_TOKEN loaded.
LLM engine ready.




Cloning the repo


In [ ]:
if os.path.exists(LOCAL_DIR):
    shutil.rmtree(LOCAL_DIR)

print(f"Cloning https://github.com/{REPO_OWNER}/{REPO_NAME}...")
repo_url = f"https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
!git clone {repo_url} --quiet
print("Repository cloned locally.")

Cloning https://github.com/tiangolo/typer...
Repository cloned locally.




> generating file tree




In [ ]:
def generate_file_tree(root_dir):
    allowed_extensions = ('.py', '.md')
    ignored_dirs = {'.git', '__pycache__', '.github', 'docs', 'tests', '.egg-info'}

    file_paths = []
    for root, dirs, files in os.walk(root_dir):
        # Modify dirs in-place to skip ignored directories
        dirs[:] = [d for d in dirs if d not in ignored_dirs]

        for file in files:
            if file.endswith(allowed_extensions):
                # Get path relative to the repo root
                relative_path = os.path.relpath(os.path.join(root, file), root_dir)
                file_paths.append(relative_path)
    return sorted(file_paths)

repo_files = generate_file_tree(LOCAL_DIR)
print(f"\n Codebase mapped! Found {len(repo_files)} source files.")
print("Sample files available to the agent:", repo_files[:5])


📂 Codebase mapped! Found 341 source files.
Sample files available to the agent: ['README.md', 'docs_src/app_dir/__init__.py', 'docs_src/app_dir/tutorial001_py310.py', 'docs_src/arguments/__init__.py', 'docs_src/arguments/default/__init__.py']




> **vectorless** langflow definition:




In [ ]:
import os # Added for GITHUB_TOKEN access
from google.colab import userdata # Added for GITHUB_TOKEN access
from langchain_openai import ChatOpenAI # Added for llm initialization
from typing import List, Optional, Literal, TypedDict
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import END, StateGraph

# --- Start LLM Initialization (Copied from vTud9ire_sno for robustness) ---
# Assuming GITHUB_MODELS_BASE is available, if not, it should be defined here.
GITHUB_MODELS_BASE = "https://models.inference.ai.azure.com"

# Ensure GITHUB_TOKEN is available in the environment.
# This block attempts to load it from Colab secrets if not already in os.environ.
if "GITHUB_TOKEN" not in os.environ:
    try:
        # This part might prompt the user if run interactively outside of initial setup.
        GITHUB_TOKEN_VAL = userdata.get('GITHUB_TOKEN')
        os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN_VAL
        print("GITHUB_TOKEN re-loaded in YavpaEkoAWpw for robustness.")
    except userdata.SecretNotFoundError:
        raise ValueError("GITHUB_TOKEN not found in Colab Secrets or environment. Please ensure the setup cell (vTud9ire_sno) has been run, or set the token.")

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key=os.environ["GITHUB_TOKEN"],
    openai_api_base=GITHUB_MODELS_BASE
)
print("LLM re-initialized in YavpaEkoAWpw.")
# --- End LLM Initialization ---

class VectorlessDecision(BaseModel):
    action: Literal["answer", "read_files"] = Field(
        ...,
        description="Choose 'answer' if your opened files have enough info. Choose 'read_files' if you need to open files from the directory tree."
    )
    reasoning: str = Field(..., description="Explain what you are looking for or how your current context answers the query.")
    files_to_open: Optional[List[str]] = Field(
        default=None,
        description="List of exact relative file paths from the provided File Tree that you want to open and read right now."
    )
    final_answer: Optional[str] = Field(default=None, description="Your comprehensive technical answer. Leave blank if opening files.")

structured_llm = llm.with_structured_output(VectorlessDecision)

# Graph State definition
class VectorlessState(TypedDict):
    query: str
    file_tree: List[str]            # List of all files in repo
    opened_files: dict              # Maps path -> file text content
    history: List[str]              # Tracks all file paths ever opened
    iterations: int
    current_decision: str
    next_files_to_read: List[str]
    final_response: str

# System Prompt for Directory Analysis
VECTORLESS_PROMPT = """You are an expert repository explorer. Your goal is to find the main entry point or class definition for the Typer application and describe how it starts. You answer questions by opening and reading local files.\nYou are given a list of all available file paths in the 'File Tree'.\n\nYOUR WORKFLOW:\n1. Carefully review the user's query and the 'AVAILABLE FILE TREE'.\n2. Identify and request to open files most relevant to the query's goal. Focus on files like `__init__.py`, `main.py`, `app.py`, or those clearly related to the 'Typer application' setup.\n3. IMPORTANT: Once a file has been opened, DO NOT request it again. Instead, thoroughly analyze its content for class definitions, main function calls, imports, and execution flow.\n4. Based on the *currently opened files*, determine if you have enough information to answer the user's query about the Typer application's main entry point and startup mechanism.\n   - If YES, choose the 'answer' action and provide your comprehensive technical summary in 'final_answer'.\n   - If NO, and if the opened files reference other crucial modules/files, request ONLY those *new*, unopened files in 'files_to_open'.\n5. If you cannot find the answer after exploring relevant files, or if you are stuck, clearly explain what information you are missing and what files you would need to examine to progress."""

#Graph Nodes
def read_files_node(state: VectorlessState):
    """Reads raw source text directly from the disk using paths requested by the LLM."""
    opened_files = state.get("opened_files", {})
    history = state.get("history", [])
    files_to_read = state.get("next_files_to_read", [])
    print(f"DEBUG: read_files_node received files_to_read: {files_to_read}") # NEW DEBUG
    print(f"DEBUG: read_files_node received opened_files: {opened_files}") # NEW DEBUG

    print(f"DEBUG: read_files_node received history: {history}") # NEW DEBUG

    current_iteration_opened_files = []

    for file_path in files_to_read:
        full_path = os.path.join(LOCAL_DIR, file_path)
        if os.path.exists(full_path):
            try:
                with open(full_path, 'r', encoding='utf-8') as f:
                    opened_files[file_path] = f.read()
                    current_iteration_opened_files.append(file_path)
                print(f"Opened and loaded file: {file_path}")
            except Exception as e:
                opened_files[file_path] = f"[Error reading file: {str(e)}]"
        else:
            opened_files[file_path] = "[File not found at specified path]"

    # Explicitly create a new history list by concatenation
    new_history = history + current_iteration_opened_files

    print(f"DEBUG: read_files_node returning history: {new_history}") # NEW DEBUG

    return {
        "opened_files": opened_files,
        "history": new_history,
        "iterations": state["iterations"] # Pass through the iteration count from the incoming state
    }

def analyze_directory_node(state: VectorlessState):
    """LLM looks at the file tree and currently read files to make a decision."""
    # Format files that have already been opened
    current_context = ""
    MAX_CHARS_PER_FILE_DISPLAY = 1000  # Max characters to display for a single opened file (Reduced from 1500)
    MAX_TOTAL_OPENED_FILE_CHARS = 4000 # Max total characters for all opened files combined (Reduced from 10000)
    total_opened_chars = 0

    for path, content in state["opened_files"].items():
        content_to_add = content
        if len(content) > MAX_CHARS_PER_FILE_DISPLAY:
            content_to_add = content[:MAX_CHARS_PER_FILE_DISPLAY] + "\n... [TRUNCATED]"

        if total_opened_chars + len(content_to_add) > MAX_TOTAL_OPENED_FILE_CHARS:
            remaining_chars = MAX_TOTAL_OPENED_FILE_CHARS - total_opened_chars
            if remaining_chars <= 0:
                break # No more space for opened file content

            content_to_add = content_to_add[:remaining_chars] + "\n... [TRUNCATED DUE TO TOTAL LIMIT]" if len(content_to_add) > remaining_chars else content_to_add
            current_context += f"\n\n=== FILE: {path} ===\n{content_to_add}\n"
            total_opened_chars += len(content_to_add)
            break # Stop adding files if total limit reached
        else:
            current_context += f"\n\n=== FILE: {path} ===\n{content_to_add}\n"
            total_opened_chars += len(content_to_add)

    # Truncate file tree for display in the prompt
    displayed_file_tree = state["file_tree"]
    MAX_FILE_TREE_PATHS_DISPLAY = 10 # Drastically reduce to 10 paths for the prompt
    if len(displayed_file_tree) > MAX_FILE_TREE_PATHS_DISPLAY:
        displayed_file_tree_start = displayed_file_tree[:MAX_FILE_TREE_PATHS_DISPLAY//2]
        displayed_file_tree_end = displayed_file_tree[-(MAX_FILE_TREE_PATHS_DISPLAY - len(displayed_file_tree_start) - 1):]
        displayed_file_tree = displayed_file_tree_start + \
                              [f"... (truncated, total {len(state['file_tree'])} files) ..."] + \
                              displayed_file_tree_end

    formatted_input = (
        f"User Query: {state['query']}\n\n"
        f"AVAILABLE FILE TREE:\n" + "\n".join(displayed_file_tree) + "\n\n"
        f"CURRENT OPENED FILE CONTENTS:\n{current_context if current_context else '[No files opened yet]'}"
        f"\n\nFILES ALREADY INSPECTED (HISTORY):\n" + "\n".join(state['history']) if state['history'] else '[No files inspected yet]'
    )

    messages = [
        SystemMessage(content=VECTORLESS_PROMPT),
        HumanMessage(content=formatted_input)
    ]

    decision = structured_llm.invoke(messages)
    print(f" ********************[Decision] Agent made a decision: {decision}")
    updates = {
        "iterations": state["iterations"] + 1,
        "current_decision": decision.action
        # IMPORTANT: Do NOT propagate 'history' or 'opened_files' here.
        # LangGraph automatically merges these if they are not explicitly returned.
        # 'read_files_node' is responsible for updating these, and we want those updates to persist.
    }

    if decision.action == "read_files":
        # Debug print to check history before filtering
        print(f"DEBUG: analyze_directory_node state['history'] before filtering: {state['history']}")
        # Filter out files that have already been opened
        files_to_open_filtered = [f for f in decision.files_to_open if f not in state["history"]]
        updates["next_files_to_read"] = files_to_open_filtered
        print(f"[Decision] Agent wants to inspect: {files_to_open_filtered}")
    else:
        updates["final_response"] = decision.final_answer
        print("[Decision] Agent found the solution and compiled the answer.")

    return updates

# Graph Assembly
def route_edge(state: VectorlessState):
    if state["iterations"] >= 10:
        print("Timeout safeguard hit.")
        return "end"
    return "continue" if state["current_decision"] == "read_files" else "end"

workflow = StateGraph(VectorlessState)
print("LOG :workflow created")
workflow.add_node("read_files", read_files_node)
print("LOG :read file node added")
workflow.add_node("analyze_directory", analyze_directory_node)
print("LOG :analyse directory node added")
workflow.set_entry_point("analyze_directory") # LLM looks at tree first
print("LOG :entry point set")
workflow.add_edge("read_files", "analyze_directory")
print("LOG :read file to analyse directory edge added")
workflow.add_conditional_edges("analyze_directory", route_edge, {"continue": "read_files", "end": END})
print("LOG :conditional edges added")
app = workflow.compile()
print(" Vector-less Repository Walker compiled successfully!")

LLM re-initialized in YavpaEkoAWpw.
LOG :workflow created
LOG :read file node added
LOG :analyse directory node added
LOG :entry point set
LOG :read file to analyse directory edge added
LOG :conditional edges added
 Vector-less Repository Walker compiled successfully!


In [ ]:
initial_state = {
    #"query": "Where is the main entry point or class definition for the Typer application defined, and how does it start?",
    "query":"Give an atomic_write_example ",
    "file_tree": repo_files,
    "opened_files": {},
    "history": [],
    "iterations": 0,
    "current_decision": "",
    "next_files_to_read": [],
    "final_response": ""
}

final_output = app.invoke(initial_state)

print("\n================== FINAL METRICS & RESPONSE ==================\n")
print(f"Total Iteration Loops: {final_output['iterations']}")
print(f"Files Inspected: {list(final_output['opened_files'].keys())}\n")
print(final_output["final_response"])

 ********************[Decision] Agent made a decision: action='read_files' reasoning="I need to find the main entry point or class definition for the Typer application, so I will start by looking for files that are likely to contain the main application logic, such as '__init__.py', 'main.py', or 'app.py'." files_to_open=['__init__.py', 'main.py', 'app.py'] final_answer=None
DEBUG: analyze_directory_node state['history'] before filtering: []
[Decision] Agent wants to inspect: ['__init__.py', 'main.py', 'app.py']
DEBUG: read_files_node received files_to_read: ['__init__.py', 'main.py', 'app.py']
DEBUG: read_files_node received opened_files: {}
DEBUG: read_files_node received history: []
DEBUG: read_files_node returning history: []
 ********************[Decision] Agent made a decision: action='read_files' reasoning='I need to find the main entry point or class definition for the Typer application, so I will start by looking for files that are likely to contain the main application logic,

In [ ]:
print(repo_files)

['README.md', 'docs_src/app_dir/__init__.py', 'docs_src/app_dir/tutorial001_py310.py', 'docs_src/arguments/__init__.py', 'docs_src/arguments/default/__init__.py', 'docs_src/arguments/default/tutorial001_an_py310.py', 'docs_src/arguments/default/tutorial001_py310.py', 'docs_src/arguments/default/tutorial002_an_py310.py', 'docs_src/arguments/default/tutorial002_py310.py', 'docs_src/arguments/envvar/__init__.py', 'docs_src/arguments/envvar/tutorial001_an_py310.py', 'docs_src/arguments/envvar/tutorial001_py310.py', 'docs_src/arguments/envvar/tutorial002_an_py310.py', 'docs_src/arguments/envvar/tutorial002_py310.py', 'docs_src/arguments/envvar/tutorial003_an_py310.py', 'docs_src/arguments/envvar/tutorial003_py310.py', 'docs_src/arguments/help/__init__.py', 'docs_src/arguments/help/tutorial001_an_py310.py', 'docs_src/arguments/help/tutorial001_py310.py', 'docs_src/arguments/help/tutorial002_an_py310.py', 'docs_src/arguments/help/tutorial002_py310.py', 'docs_src/arguments/help/tutorial003_an_